<a href="https://colab.research.google.com/github/yunioktasafitri/Eksplorasi-Mahadata/blob/main/123140213_YuniOktaSafitri_Eksplorasi_Mahadata.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **EKSPLORASI MAHADATA**
Nama: Yuni Okta Safitri

NIM: 123140213

##**1. Spesifikasi Colab**

In [1]:
   !nproc
   !free -h
   !df -h

2
               total        used        free      shared  buff/cache   available
Mem:            12Gi       883Mi       8.9Gi       2.2Mi       3.2Gi        11Gi
Swap:             0B          0B          0B
Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   21G   88G  20% /
tmpfs            64M     0   64M   0% /dev
shm             5.7G     0  5.7G   0% /dev/shm
/dev/root       2.0G  1.3G  696M  65% /usr/sbin/docker-init
tmpfs           6.4G   32K  6.4G   1% /var/colab
/dev/sda1       114G   22G   93G  19% /kaggle/input
tmpfs           6.4G     0  6.4G   0% /proc/acpi
tmpfs           6.4G     0  6.4G   0% /proc/scsi
tmpfs           6.4G     0  6.4G   0% /sys/firmware


In [2]:
!lsb_release -a

No LSB modules are available.
Distributor ID:	Ubuntu
Description:	Ubuntu 24.04.5 LTS
Release:	24.04
Codename:	noble


In [3]:
!apt-get install -y -qq openjdk-11-jre-headless > /dev/null
!java -version
!curl -L -o ycsb.tar.gz https://github.com/brianfrankcooper/YCSB/releases/download/0.17.0/ycsb-0.17.0.tar.gz
!tar xzf ycsb.tar.gz
!ls ycsb-0.17.0

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  675M  100  675M    0     0  21.6M      0  0:00:31  0:00:31 --:--:-- 15.6M
accumulo1.6-binding	   geode-binding	    memcached-binding
accumulo1.7-binding	   googlebigtable-binding   mongodb-binding
accumulo1.8-binding	   googledatastore-bindin

In [4]:
!curl -L -o mongo.tgz https://fastdl.mongodb.org/linux/mongodb-linux-x86_64-ubuntu2204-6.0.16.tgz
!tar xzf mongo.tgz
!mkdir -p /content/mongo-data
!/content/mongodb-linux-x86_64-ubuntu2204-6.0.16/bin/mongod --dbpath /content/mongo-data --fork --logpath /content/mongod.log --bind_ip 127.0.0.1
!/content/mongodb-linux-x86_64-ubuntu2204-6.0.16/bin/mongod --version | head -1

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 69.6M  100 69.6M    0     0   124M      0 --:--:-- --:--:-- --:--:--  124M
about to fork child process, waiting until server is ready for connections.
forked process: 4493
child process started successfully, parent exiting
db version v6.0.16


In [9]:
%cd /content/ycsb-0.17.0
CP = "mongodb-binding/lib/*:lib/*"
P = "-p recordcount=10000 -p operationcount=10000 -p fieldcount=20 -p fieldlength=500 -p mongodb.url=mongodb://localhost:27017/ycsb"
CLS = "site.ycsb.Client -db site.ycsb.db.MongoDbClient -s -P workloads/workloadc"

!java -cp "{CP}" {CLS} -load {P} -threads 1 2>&1 | grep -E "RunTime|Throughput|Exception|Error"
!java -cp "{CP}" {CLS} -t    {P} -threads 1 2>&1 | grep -E "RunTime|Throughput|Exception|Error"

/content/ycsb-0.17.0
[OVERALL], RunTime(ms), 8916
[OVERALL], Throughput(ops/sec), 1121.5791834903544
[OVERALL], RunTime(ms), 6311
[OVERALL], Throughput(ops/sec), 1584.534938995405


In [10]:
!pip -q install pymongo
G = """workload=site.ycsb.workloads.CoreWorkload
readallfields=true
readproportion=0.05
updateproportion=0.95
scanproportion=0
insertproportion=0
requestdistribution=zipfian
"""
H = G.replace("readproportion=0.05","readproportion=0").replace("updateproportion=0.95","updateproportion=1.0")
open("/content/ycsb-0.17.0/workloads/workloadg","w").write(G)
open("/content/ycsb-0.17.0/workloads/workloadh","w").write(H)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.8/354.8 kB 23.4 MB/s eta 0:00:00


163

In [14]:
import subprocess, re, os, csv, time, pymongo
os.chdir("/content/ycsb-0.17.0")
CP  = "mongodb-binding/lib/*:lib/*"
OUT = "/content/hasil_mongodb.csv"
client = pymongo.MongoClient("mongodb://localhost:27017")

def ycsb(phase, wl, n, th):
    cmd = (f'java -cp "{CP}" site.ycsb.Client -db site.ycsb.db.MongoDbClient -s '
           f'-P workloads/{wl} {phase} -p recordcount={n} -p operationcount={n} '
           f'-p fieldcount=20 -p fieldlength=500 '
           f'-p mongodb.url=mongodb://localhost:27017/ycsb -threads {th}')
    out = subprocess.run(cmd, shell=True, text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT).stdout
    m = re.search(r"\[OVERALL\], RunTime\(ms\), ([\d.]+)", out)
    return float(m.group(1))/1000 if m else None

In [16]:
RECORDS   = [10000]
THREADS   = [1, 2, 4]
REPS      = 3
WORKLOADS = ["workloada","workloadb","workloadc","workloadd",
             "workloade","workloadf","workloadg","workloadh"]

In [17]:
done = set()
if os.path.exists(OUT):
    for r in csv.DictReader(open(OUT)):
        done.add((r["workload"], r["records"], r["threads"], r["rep"]))
else:
    open(OUT,"w").write("db,workload,records,threads,rep,runtime_s\n")

for n in RECORDS:
  for wl in WORKLOADS:
    for th in THREADS:
      for rep in range(1, REPS+1):
        if (wl, str(n), str(th), str(rep)) in done: continue
        client.drop_database("ycsb")            # data bersih tiap run
        ycsb("-load", wl, n, 2)                 # fase load tidak dihitung
        t = ycsb("-t", wl, n, th)               # fase run: ini yang diukur
        open(OUT,"a").write(f"mongodb,{wl},{n},{th},{rep},{t}\n")
        print(wl, n, th, rep, t, flush=True)

workloada 10000 1 1 9.218
workloada 10000 1 2 9.417
workloada 10000 1 3 7.638
workloada 10000 2 1 6.401
workloada 10000 2 2 6.124
workloada 10000 2 3 6.987
workloada 10000 4 1 6.271
workloada 10000 4 2 6.169
workloada 10000 4 3 5.994
workloadb 10000 1 1 7.309
workloadb 10000 1 2 6.617
workloadb 10000 1 3 6.461
workloadb 10000 2 1 6.034
workloadb 10000 2 2 6.579
workloadb 10000 2 3 5.852
workloadb 10000 4 1 6.321
workloadb 10000 4 2 6.451
workloadb 10000 4 3 6.219
workloadc 10000 1 1 6.546
workloadc 10000 1 2 6.369
workloadc 10000 1 3 6.471
workloadc 10000 2 1 5.792
workloadc 10000 2 2 5.919
workloadc 10000 2 3 6.679
workloadc 10000 4 1 5.864
workloadc 10000 4 2 5.923
workloadc 10000 4 3 5.92
workloadd 10000 1 1 6.929
workloadd 10000 1 2 7.21
workloadd 10000 1 3 6.614
workloadd 10000 2 1 6.569
workloadd 10000 2 2 7.669
workloadd 10000 2 3 9.79
workloadd 10000 4 1 8.292
workloadd 10000 4 2 8.489
workloadd 10000 4 3 9.02
workloade 10000 1 1 25.4
workloade 10000 1 2 24.755
workloade 10000 

In [20]:
print(OUT)
!ls -la

/content/hasil_mongodb.csv
total 232
drwxr-xr-x 54 root root   4096 Oct 13 17:36 .
drwxr-xr-x  1 root root   4096 Oct 13 17:43 ..
drwxr-xr-x  4 root root   4096 Oct 13 17:36 accumulo1.6-binding
drwxr-xr-x  4 root root   4096 Oct 13 17:36 accumulo1.7-binding
drwxr-xr-x  4 root root   4096 Oct 13 17:36 accumulo1.8-binding
drwxr-xr-x  3 root root   4096 Oct 13 17:36 aerospike-binding
drwxr-xr-x  3 root root   4096 Oct 13 17:36 arangodb-binding
drwxr-xr-x  3 root root   4096 Oct 13 17:36 asynchbase-binding
drwxr-xr-x  3 root root   4096 Oct 13 17:36 azurecosmos-binding
drwxr-xr-x  3 root root   4096 Oct 13 17:36 azuretablestorage-binding
drwxr-xr-x  2 root root   4096 Oct 13 17:38 bin
drwxr-xr-x  3 root root   4096 Oct 13 17:36 cassandra-binding
drwxr-xr-x  3 root root   4096 Oct 13 17:36 cloudspanner-binding
drwxr-xr-x  3 root root   4096 Oct 13 17:36 couchbase2-binding
drwxr-xr-x  3 root root   4096 Oct 13 17:36 couchbase-binding
drwxr-xr-x  3 root root   4096 Oct 13 17:36 crail-binding


In [21]:
import pandas as pd
df = pd.read_csv(OUT)
print(len(df), "baris")
print(df.groupby(["records","workload","threads"]).size().value_counts())

72 baris
3    24
Name: count, dtype: int64


In [22]:
print(OUT)

/content/hasil_mongodb.csv


In [23]:
!pkill mongod

## **3. CouchDB**

In [25]:
%%bash
export DEBIAN_FRONTEND=noninteractive
cat <<'EOF' | debconf-set-selections
couchdb couchdb/mode select standalone
couchdb couchdb/mode seen true
couchdb couchdb/cookie string ycsbcookie123
couchdb couchdb/cookie seen true
couchdb couchdb/bindaddress string 127.0.0.1
couchdb couchdb/bindaddress seen true
couchdb couchdb/adminpass password ycsbpass
couchdb couchdb/adminpass seen true
couchdb couchdb/adminpass_again password ycsbpass
couchdb couchdb/adminpass_again seen true
EOF
dpkg --configure -a
apt-get install -y -qq couchdb
ls /opt/couchdb/bin

Setting up couchdb (3.5.3~noble) ...
couchdb
couchdb.cmd
couchjs
remsh
weatherreport


invoke-rc.d: could not determine current runlevel
invoke-rc.d: policy-rc.d denied execution of start.
Created symlink /etc/systemd/system/multi-user.target.wants/couchdb.service → /usr/lib/systemd/system/couchdb.service.
CouchDB may not have started correctly (no init?)
Once CouchDB has started correctly, run the following:

  curl -X PUT --user '<admin-user>:<admin-pass>' http://127.0.0.1:5984/_users
  curl -X PUT --user '<admin-user>:<admin-pass>' http://127.0.0.1:5984/_replicator



In [26]:
%%bash
su couchdb -s /bin/bash -c 'nohup /opt/couchdb/bin/couchdb > /tmp/couchdb.log 2>&1 &'
sleep 20
curl -s http://127.0.0.1:5984/
echo
curl -s -X PUT -u admin:ycsbpass http://127.0.0.1:5984/_users
curl -s -X PUT -u admin:ycsbpass http://127.0.0.1:5984/_replicator
echo
tail -n 5 /tmp/couchdb.log

{"couchdb":"Welcome","version":"3.5.3","git_sha":"7a1f97558","uuid":"aedbb782c66e4a1e1d18209e7823b2c0","features":["access-ready","partitioned","pluggable-storage-engines","reshard","scheduler"],"vendor":{"name":"The Apache Software Foundation"}}

{"ok":true}
{"ok":true}



In [27]:
!cat ycsb-0.17.0/workloads/workloada | grep -v '^#'
!cat ycsb-0.17.0/workloads/workloadg | grep -v '^#'
!cat ycsb-0.17.0/workloads/workloadh | grep -v '^#'
import inspect; print(inspect.getsource(ycsb))

cat: ycsb-0.17.0/workloads/workloada: No such file or directory
cat: ycsb-0.17.0/workloads/workloadg: No such file or directory
cat: ycsb-0.17.0/workloads/workloadh: No such file or directory
def ycsb(phase, wl, n, th):
    cmd = (f'java -cp "{CP}" site.ycsb.Client -db site.ycsb.db.MongoDbClient -s '
           f'-P workloads/{wl} {phase} -p recordcount={n} -p operationcount={n} '
           f'-p fieldcount=20 -p fieldlength=500 '
           f'-p mongodb.url=mongodb://localhost:27017/ycsb -threads {th}')
    out = subprocess.run(cmd, shell=True, text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT).stdout
    m = re.search(r"\[OVERALL\], RunTime\(ms\), ([\d.]+)", out)
    return float(m.group(1))/1000 if m else None



In [28]:
!pwd
!cat workloads/workloada | grep -v '^#'
!echo ----- G
!cat workloads/workloadg | grep -v '^#'
!echo ----- H
!cat workloads/workloadh | grep -v '^#'

/content/ycsb-0.17.0



recordcount=1000
operationcount=1000
workload=site.ycsb.workloads.CoreWorkload

readallfields=true

readproportion=0.5
updateproportion=0.5
scanproportion=0
insertproportion=0

requestdistribution=zipfian

----- G
workload=site.ycsb.workloads.CoreWorkload
readallfields=true
readproportion=0.05
updateproportion=0.95
scanproportion=0
insertproportion=0
requestdistribution=zipfian
----- H
workload=site.ycsb.workloads.CoreWorkload
readallfields=true
readproportion=0
updateproportion=1.0
scanproportion=0
insertproportion=0
requestdistribution=zipfian
